## Harness Engineering

The model reasons. The harness does everything else: it assembles the context, runs the
tools the model asks for, checks the result against evidence rather than the model's own
confidence, compacts its history when it grows too long, and decides when to stop.

The loop below is small enough to read in one sitting and real in every other respect —
it calls an actual model, writes actual files, and runs an actual test suite in a
subprocess.

In [ ]:
#%pip install --quiet openai python-dotenv

In [ ]:
OPENAI="gpt-4o-mini"

import os
from dotenv import load_dotenv
load_dotenv("../keys.env")
assert os.environ["OPENAI_API_KEY"][:2] == "sk",\
       "Please specify the OPENAI_API_KEY access token in keys.env file"

In [ ]:
import json
import pathlib
import shutil
import subprocess
import sys

from openai import OpenAI

## The workspace and the spec

The harness hands the model a scratch directory and a test suite. The tests are the
specification: the harness will *run* them, not ask the model whether they pass.

`SKILLS` is the other half of what gets injected — house rules the model would otherwise
have to guess at. A real harness keeps many, and selects the ones the task needs.

In [ ]:
WORKSPACE = pathlib.Path("workspace")

SKILLS = {"python": "Write plain functions. No classes. Type-annotate."}

SPEC = '''\
import unittest
from slugify import slugify


class TestSlugify(unittest.TestCase):
    def test_basic(self):
        self.assertEqual(slugify("Hello World"), "hello-world")

    def test_punctuation(self):
        self.assertEqual(slugify("Hello, World!"), "hello-world")

    def test_collapses_space(self):
        self.assertEqual(slugify("  Hello   World  "), "hello-world")
'''

## The model: reasoning only

Everything else is the harness. Like every provider call it is stateless — each turn it
reads the context it was handed and nothing else. The harness owns what goes into that
context, which is the whole job.

`tools` is optional because one caller deliberately omits it: the summarizer below should
summarize, not act.

In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def llm(context, tools=None):
    """Return the model's next message: a tool call, or plain text."""
    reply = client.chat.completions.create(
        model=OPENAI, messages=context, **({"tools": tools} if tools else {}))
    return reply.choices[0].message

## Context injection

What the model sees, rebuilt from scratch on every single call: the system prompt, the
applicable skill, the task, and the history so far. Nothing persists on the model's side —
if it isn't in this list, the model does not know it.

In [ ]:
def assemble(task, history):
    system = "You write Python to satisfy the given test."
    return [{"role": "system", "content": f"{system}\n{SKILLS['python']}"},
            {"role": "user", "content": task}] + history

## Interacting with the real world

The model requests; the harness acts. Two capabilities are enough for this task: writing a
file into the workspace, and running the tests.

In [ ]:
def write_file(name, body):
    (WORKSPACE / name).write_text(body)
    return f"wrote {WORKSPACE / name}"


def run_tests():
    proc = subprocess.run(
        [sys.executable, "-m", "unittest", "discover",
         "-s", str(WORKSPACE), "-t", str(WORKSPACE)],
        capture_output=True, text=True)
    return proc.returncode, proc.stderr.strip()

## The tool contract

Two halves that must be kept in step: `TOOLS` is what the harness will actually run,
`TOOL_SCHEMA` is what the model is allowed to request. The harness answers only for names
it recognizes.

Note that errors are *returned*, not raised. A malformed argument or a bad tool name comes
back to the model as text it can read and correct on the next turn — an exception here
would end the run instead.

In [ ]:
TOOLS = {"write_file": write_file}          # what the harness will run

TOOL_SCHEMA = [{                            # what the model may request
    "type": "function",
    "function": {
        "name": "write_file",
        "description": "Write a file into the workspace, overwriting it.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string",
                         "description": "File name in the workspace."},
                "body": {"type": "string",
                         "description": "Full contents of the file."},
            },
            "required": ["name", "body"],
        },
    },
}]


def execute(call):
    fn = TOOLS.get(call.function.name)      # a permission check would go here
    if fn is None:
        return f"error: no such tool {call.function.name}"
    try:
        return fn(**json.loads(call.function.arguments))  # a real harness
    except Exception as exc:                              # sandboxes this
        return f"error: {exc}"              # the model sees this and can retry

## Organize: compaction

Replace the history with a shorter representation of it. What survives is chosen
deliberately — the objective, the dead ends and why they were dead, and the paths on disk.
The files themselves stay on disk, so the summary does not need to carry them.

In [ ]:
def compact(history):
    prompt = ("Summarize for a fresh context: the current objective, the "
              "approaches already ruled out and why, and the paths of files "
              "already written to disk.")
    reply = llm(history + [{"role": "user", "content": prompt}])  # no tools:
    return [{"role": "assistant", "content": reply.content}]      # summarize

## Check and verify

Evidence, not the model's confidence. The model saying "done" is a claim; the exit code of
the test run is the fact that settles it.

In [ ]:
def verify():
    code, output = run_tests()
    return code == 0, output

## The agent loop

Reason, act, verify, repeat — bounded by a step limit and a context budget.

Three details are forced by the API rather than chosen. The assistant message goes back
into the history verbatim, because a `tool` message is only valid directly after the
assistant turn that requested it. Every tool call in a message gets its own answer, keyed
by `tool_call_id`. And the test failure comes back as a `user` message, because the
"I'm done" reply carries no tool call for it to answer.

In [ ]:
def run(task, max_steps=10, budget=2000):
    history = []
    for step in range(1, max_steps + 1):
        size = sum(len(json.dumps(m)) for m in history)   # chars, not tokens
        if size > budget:                                 # lower it to watch
            history = compact(history)                    # compaction fire
            print(f"[{step}] compacted {size} chars of history")

        reply = llm(assemble(task, history), TOOL_SCHEMA)    # model reasons
        history.append(reply.model_dump(exclude_none=True))  # the API wants
                                                             # its own turn back
        if reply.tool_calls:
            for call in reply.tool_calls:                 # answer every one
                result = execute(call)                    # action
                print(f"[{step}] {call.function.name}: {result}")
                history.append({"role": "tool", "tool_call_id": call.id,
                                "content": result})
            continue

        passed, evidence = verify()                       # check before accepting
        print(f"[{step}] model says done; tests {'PASS' if passed else 'FAIL'}")
        if passed:
            return reply.content
        history.append({"role": "user",
                        "content": f"TEST_FAILURE {evidence}"})

    return "Stopped: step limit reached."                 # error recovery

## Run it

The workspace is wiped first so the run starts from the spec and nothing else. Everything
that appears under `workspace/` after this cell was put there by the model.

In [ ]:
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir(parents=True)
(WORKSPACE / "test_slugify.py").write_text(SPEC)      # the spec is the input

print(run("Implement slugify(text) in slugify.py so this test suite "
          f"passes:\n\n{SPEC}"))

## What the model wrote

The harness already proved this passes. Reading it is for us, not for the loop.

Two paths stay dormant on a good run, and both are worth provoking: lower `budget` to see
compaction fire, and tighten the spec (a `test_unicode` case, say) to see a failure come
back as `TEST_FAILURE` and get fixed on the next turn.

In [ ]:
print((WORKSPACE / "slugify.py").read_text())